# Model Research Pipeline for US30 Time Series Forecasting with ML, DL, and RL

**Goal:** Load US30 (^DJI) data, preprocess, engineer features (lags, technical indicators), train ML/DL/RL models, evaluate with walk-forward CV and compare (MAE, RMSE, Sharpe).

**Colab + SSH:** Run this notebook on Colab (GPU) via Cursor Remote-SSH or upload to [colab.research.google.com](https://colab.research.google.com).

## 1. Environment Setup

In [ ]:
# Install packages (Colab-specific; run once per runtime)
!pip install -q yfinance pandas_ta tensorflow stable-baselines3 gym scikit-learn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import gym
from gym import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import matplotlib.pyplot as plt
import pandas_ta as ta

# Optional: Mount Google Drive for saving outputs (Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT = "/content/drive/MyDrive/colab_us30_models"
except Exception:
    SAVE_ROOT = "./colab_us30_models"

import os
os.makedirs(SAVE_ROOT, exist_ok=True)
print("Setup OK. Save root:", SAVE_ROOT)

## 2. Data Loading and Preprocessing

In [ ]:
def load_data(ticker="^DJI", start="2010-01-01", end="2026-03-01"):
    data = yf.download(ticker, start=start, end=end, progress=False)
    if data.columns.nlevels > 1:
        data.columns = data.columns.get_level_values(0)
    data["Returns"] = data["Close"].pct_change()
    return data.dropna()

data = load_data()
print(data.head())
print("Shape:", data.shape)

In [ ]:
def preprocess_data(df):
    # Anomaly detection: remove outliers (z-score > 3 on Returns)
    r = df["Returns"]
    df = df[np.abs(r - r.mean()) <= (3 * r.std())].copy()
    # Normalize numeric columns
    cols = df.select_dtypes(include=[np.number]).columns.tolist()
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(
        scaler.fit_transform(df[cols]),
        columns=cols,
        index=df.index
    )
    return df_scaled, scaler

data_scaled, scaler = preprocess_data(data)
print("Preprocessed shape:", data_scaled.shape)

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    close = df["Close"]
    # Lags
    for lag in [1, 5, 10]:
        df[f"Close_lag_{lag}"] = close.shift(lag)
    # Technical indicators (pandas_ta)
    df["SMA_10"] = ta.sma(close, length=10)
    df["RSI_14"] = ta.rsi(close, length=14)
    macd_df = ta.macd(close, fast=12, slow=26, signal=9)
    if macd_df is not None and not macd_df.empty:
        macd_col = [c for c in macd_df.columns if c.startswith("MACD_") and "s_" not in c and "h_" not in c]
        if macd_col:
            df["MACD"] = macd_df[macd_col[0]]
        else:
            df["MACD"] = macd_df.iloc[:, 0]
    return df.dropna()

features = engineer_features(data_scaled)
feature_cols = [c for c in features.columns if c != "Close"]
X = features[feature_cols]
y = features["Close"].shift(-1).dropna()
X = X.iloc[:-1].loc[y.index]
print("X shape:", X.shape, "y shape:", y.shape)

## 4. Model Training and Evaluation Pipeline (Walk-Forward CV)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

def evaluate_ml(model, X, y):
    results = []
    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        returns = pd.Series(preds).pct_change().dropna()
        sharpe = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() != 0 else 0
        results.append({"MAE": mae, "RMSE": rmse, "Sharpe": sharpe})
    return pd.DataFrame(results).mean()

def evaluate_lstm(model, X, y, epochs=50, batch_size=32):
    results = []
    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx].values.reshape((X.iloc[train_idx].shape[0], 1, X.shape[1]))
        X_test = X.iloc[test_idx].values.reshape((X.iloc[test_idx].shape[0], 1, X.shape[1]))
        y_train = y.iloc[train_idx].values
        y_test = y.iloc[test_idx].values
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        preds = model.predict(X_test, verbose=0).flatten()
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        returns = pd.Series(preds).pct_change().dropna()
        sharpe = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() != 0 else 0
        results.append({"MAE": mae, "RMSE": rmse, "Sharpe": sharpe})
    return pd.DataFrame(results).mean()

In [ ]:
%%time
# ML: Random Forest and SVM
rf_model = RandomForestRegressor(n_estimators=100)
svm_model = SVR(kernel="rbf")
rf_results = evaluate_ml(rf_model, X, y)
svm_results = evaluate_ml(svm_model, X, y)
print("RF:", rf_results)
print("SVM:", svm_results)

In [ ]:
# DL: LSTM
def build_lstm(input_shape):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=input_shape),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

lstm_model = build_lstm((1, X.shape[1]))
lstm_results = evaluate_lstm(lstm_model, X, y, epochs=50, batch_size=32)
print("LSTM:", lstm_results)

In [ ]:
# RL: Trading environment and PPO
class TradingEnv(gym.Env):
    def __init__(self, df):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.action_space = spaces.Discrete(3)  # 0=sell, 1=hold, 2=buy
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(len(self.df.columns),),
            dtype=np.float32
        )
        self.current_step = 0

    def reset(self):
        self.current_step = 0
        return self.df.iloc[self.current_step].values.astype(np.float32)

    def step(self, action):
        reward = 0.0
        if "Returns" in self.df.columns:
            reward = self.df["Returns"].iloc[self.current_step] * (action - 1)
        self.current_step += 1
        done = self.current_step >= len(self.df) - 1
        obs = self.df.iloc[min(self.current_step, len(self.df) - 1)].values.astype(np.float32)
        return obs, float(reward), done, {}

env = DummyVecEnv([lambda: TradingEnv(features)])
rl_model = PPO("MlpPolicy", env, verbose=0)
rl_model.learn(total_timesteps=10000)

def evaluate_rl(model, env, n_episodes=5):
    rewards_list = []
    for _ in range(n_episodes):
        obs = env.reset()
        done = False
        while not done:
            action, _ = model.predict(obs)
            obs, reward, done, _ = env.step(action)
            rewards_list.append(reward[0])
    rewards = np.array(rewards_list)
    sharpe = (np.mean(rewards) / np.std(rewards) * np.sqrt(252)) if np.std(rewards) != 0 else 0
    return pd.Series({"MAE": np.nan, "RMSE": np.nan, "Sharpe": sharpe})

rl_results = evaluate_rl(rl_model, env)
print("PPO RL:", rl_results)

## 5. Comparison and Insights

In [ ]:
comparison = pd.DataFrame({
    "Random Forest": rf_results,
    "SVM": svm_results,
    "LSTM": lstm_results,
    "PPO RL": rl_results
}).T
print(comparison)
comparison.to_csv(os.path.join(SAVE_ROOT, "us30_comparison.csv"))

In [ ]:
# Plot: last fold predictions vs actual (using LSTM as example)
train_idx, test_idx = list(tscv.split(X))[-1]
X_test = X.iloc[test_idx].values.reshape((len(test_idx), 1, X.shape[1]))
y_test = y.iloc[test_idx].values
preds = lstm_model.predict(X_test, verbose=0).flatten()

plt.figure(figsize=(10, 5))
plt.plot(y_test, label="Actual")
plt.plot(preds, label="Predicted (LSTM)")
plt.legend()
plt.title("US30 Close (last CV fold)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_ROOT, "us30_lstm_pred.png"), dpi=150)
plt.show()

## 6. Next Steps (Save Models, Hyperparameter Tuning)

- Save best model: `model.save(...)` or `lstm_model.save(...)`.
- Hyperparameter tuning: GridSearchCV with TimeSeriesSplit.
- MLOps: e.g. mlflow logging.

In [ ]:
# Save models to Drive or local
lstm_model.save(os.path.join(SAVE_ROOT, "us30_lstm.keras"))
rl_model.save(os.path.join(SAVE_ROOT, "us30_ppo_rl.zip"))
print("Models saved to", SAVE_ROOT)